In [ ]:
!pip install boto3 langgraph langchain langchain-aws pydantic python-dotenv

In [ ]:
!pip install fastapi uvicorn

In [2]:
import os

os.environ["AWS_REGION"] = "us-east-1"
os.environ["BEDROCK_MODEL_ID"] = "amazon.nova-lite-v1:0"

In [3]:
os.environ["AWS_ACCESS_KEY_ID"] = "Your-Access-ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "Your-secret-KEY"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

In [4]:
from typing import Optional, List, Literal
from pydantic import BaseModel


class ClaimRequest(BaseModel):
    claim_id: str
    customer_id: str
    policy_id: str
    claim_type: str
    claim_amount: float
    incident_description: str
    incident_location: str
    previous_claims_count: int


class ClaimState(BaseModel):
    claim: ClaimRequest
    claim_summary: Optional[str] = None
    policy_status: Optional[Literal["VALID", "INVALID", "NEEDS_REVIEW"]] = None
    fraud_status: Optional[Literal["LOW", "MEDIUM", "HIGH"]] = None
    risk_score: Optional[int] = None
    decision: Optional[Literal["APPROVED", "REJECTED", "MANUAL_REVIEW"]] = None
    reasons: List[str] = []
    final_response: Optional[str] = None

In [5]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model=os.environ["BEDROCK_MODEL_ID"],
    region_name=os.environ["AWS_REGION"],
    temperature=0.2,
    max_tokens=1000,
)

In [6]:
def validate_policy_tool(claim: ClaimRequest):
    invalid_policy_ids = ["POL-000", "POL-999"]

    if claim.policy_id in invalid_policy_ids:
        return {
            "status": "INVALID",
            "reason": "Policy is inactive or expired."
        }

    if claim.claim_amount > 100000:
        return {
            "status": "NEEDS_REVIEW",
            "reason": "Claim amount is above automatic approval limit."
        }

    return {
        "status": "VALID",
        "reason": "Policy is active and claim amount is within allowed range."
    }


def fraud_check_tool(claim: ClaimRequest):
    fraud_reasons = []
    fraud_score = 0

    if claim.previous_claims_count >= 5:
        fraud_score += 40
        fraud_reasons.append("Customer has high number of previous claims.")

    if claim.claim_amount > 75000:
        fraud_score += 30
        fraud_reasons.append("Claim amount is unusually high.")

    suspicious_keywords = ["stolen", "fire", "cash", "unknown person"]

    for word in suspicious_keywords:
        if word.lower() in claim.incident_description.lower():
            fraud_score += 10
            fraud_reasons.append(
                f"Incident description contains suspicious keyword: {word}"
            )

    if fraud_score >= 60:
        status = "HIGH"
    elif fraud_score >= 30:
        status = "MEDIUM"
    else:
        status = "LOW"

    return {
        "status": status,
        "fraud_score": fraud_score,
        "reasons": fraud_reasons or ["No major fraud indicators found."]
    }


def risk_score_tool(claim: ClaimRequest, policy_status: str, fraud_status: str):
    score = 20

    if claim.claim_amount > 50000:
        score += 20

    if claim.previous_claims_count >= 3:
        score += 20

    if policy_status == "NEEDS_REVIEW":
        score += 20

    if policy_status == "INVALID":
        score += 50

    if fraud_status == "MEDIUM":
        score += 20

    if fraud_status == "HIGH":
        score += 50

    return min(score, 100)

In [7]:
from langgraph.graph import StateGraph, END


def understand_claim_node(state: ClaimState):
    claim = state.claim

    prompt = f"""
    You are an insurance claim understanding agent.

    Summarize the following insurance claim in simple business language.

    Claim ID: {claim.claim_id}
    Customer ID: {claim.customer_id}
    Policy ID: {claim.policy_id}
    Claim Type: {claim.claim_type}
    Claim Amount: {claim.claim_amount}
    Incident Location: {claim.incident_location}
    Previous Claims Count: {claim.previous_claims_count}
    Incident Description: {claim.incident_description}

    Return only a concise summary.
    """

    response = llm.invoke(prompt)

    return {
        "claim_summary": response.content,
        "reasons": state.reasons + ["Claim understanding completed."]
    }


def policy_validation_node(state: ClaimState):
    result = validate_policy_tool(state.claim)

    return {
        "policy_status": result["status"],
        "reasons": state.reasons + [result["reason"]]
    }


def fraud_check_node(state: ClaimState):
    result = fraud_check_tool(state.claim)

    return {
        "fraud_status": result["status"],
        "reasons": state.reasons + result["reasons"]
    }


def risk_scoring_node(state: ClaimState):
    score = risk_score_tool(
        claim=state.claim,
        policy_status=state.policy_status,
        fraud_status=state.fraud_status
    )

    return {
        "risk_score": score,
        "reasons": state.reasons + [f"Risk score calculated as {score}/100."]
    }


def decision_node(state: ClaimState):
    if state.policy_status == "INVALID":
        decision = "REJECTED"
        reason = "Claim rejected because policy is invalid."

    elif state.fraud_status == "HIGH":
        decision = "MANUAL_REVIEW"
        reason = "Claim sent for manual review due to high fraud risk."

    elif state.risk_score is not None and state.risk_score >= 70:
        decision = "MANUAL_REVIEW"
        reason = "Claim sent for manual review because risk score is high."

    elif state.policy_status == "VALID" and state.fraud_status == "LOW":
        decision = "APPROVED"
        reason = "Claim approved because policy is valid and fraud risk is low."

    else:
        decision = "MANUAL_REVIEW"
        reason = "Claim requires manual review due to medium risk indicators."

    return {
        "decision": decision,
        "reasons": state.reasons + [reason]
    }


def final_response_node(state: ClaimState):
    prompt = f"""
    You are an insurance claim decision assistant.

    Prepare a professional final response for the claim processing team.

    Claim Summary:
    {state.claim_summary}

    Policy Status:
    {state.policy_status}

    Fraud Status:
    {state.fraud_status}

    Risk Score:
    {state.risk_score}

    Final Decision:
    {state.decision}

    Reasons:
    {state.reasons}

    Write the response in this structure:
    1. Claim Summary
    2. Validation Result
    3. Fraud Assessment
    4. Risk Score
    5. Final Decision
    6. Explanation
    """

    response = llm.invoke(prompt)

    return {
        "final_response": response.content
    }

In [8]:
def build_claim_graph():
    graph = StateGraph(ClaimState)

    graph.add_node("understand_claim", understand_claim_node)
    graph.add_node("policy_validation", policy_validation_node)
    graph.add_node("fraud_check", fraud_check_node)
    graph.add_node("risk_scoring", risk_scoring_node)
    graph.add_node("decision", decision_node)
    graph.add_node("final_response", final_response_node)

    graph.set_entry_point("understand_claim")

    graph.add_edge("understand_claim", "policy_validation")
    graph.add_edge("policy_validation", "fraud_check")
    graph.add_edge("fraud_check", "risk_scoring")
    graph.add_edge("risk_scoring", "decision")
    graph.add_edge("decision", "final_response")
    graph.add_edge("final_response", END)

    return graph.compile()


claim_graph = build_claim_graph()

In [9]:
claim = ClaimRequest(
    claim_id="CLM-1001",
    customer_id="CUS-9001",
    policy_id="POL-123",
    claim_type="Vehicle Damage",
    claim_amount=45000,
    incident_description="Customer reported vehicle damage after accident on highway.",
    incident_location="Bangalore",
    previous_claims_count=1
)

initial_state = ClaimState(claim=claim)

result = claim_graph.invoke(initial_state)

result

{'claim': ClaimRequest(claim_id='CLM-1001', customer_id='CUS-9001', policy_id='POL-123', claim_type='Vehicle Damage', claim_amount=45000.0, incident_description='Customer reported vehicle damage after accident on highway.', incident_location='Bangalore', previous_claims_count=1),
 'claim_summary': 'Claim ID: CLM-1001  \nCustomer ID: CUS-9001  \nPolicy ID: POL-123  \nClaim Type: Vehicle Damage  \nClaim Amount: $45,000.00  \nLocation: Bangalore  \nPrevious Claims: 1  \nDescription: Vehicle damaged in a highway accident.',
 'policy_status': 'VALID',
 'fraud_status': 'LOW',
 'risk_score': 20,
 'decision': 'APPROVED',
 'reasons': ['Claim understanding completed.',
  'Policy is active and claim amount is within allowed range.',
  'No major fraud indicators found.',
  'Risk score calculated as 20/100.',
  'Claim approved because policy is valid and fraud risk is low.'],
 'final_response': '**Claim Summary**  \nClaim ID: CLM-1001  \nCustomer ID: CUS-9001  \nPolicy ID: POL-123  \nClaim Type: Ve

In [10]:
print("Claim ID:", result["claim"].claim_id)
print("Decision:", result["decision"])
print("Policy Status:", result["policy_status"])
print("Fraud Status:", result["fraud_status"])
print("Risk Score:", result["risk_score"])

print("\nReasons:")
for reason in result["reasons"]:
    print("-", reason)

print("\nFinal Response:")
print(result["final_response"])

Claim ID: CLM-1001
Decision: APPROVED
Policy Status: VALID
Fraud Status: LOW
Risk Score: 20

Reasons:
- Claim understanding completed.
- Policy is active and claim amount is within allowed range.
- No major fraud indicators found.
- Risk score calculated as 20/100.
- Claim approved because policy is valid and fraud risk is low.

Final Response:
**Claim Summary**  
Claim ID: CLM-1001  
Customer ID: CUS-9001  
Policy ID: POL-123  
Claim Type: Vehicle Damage  
Claim Amount: $45,000.00  
Location: Bangalore  
Previous Claims: 1  
Description: Vehicle damaged in a highway accident.

---

**Validation Result**  
The policy associated with this claim (Policy ID: POL-123) is currently active and valid. The claim amount of $45,000.00 falls within the allowable range as per the policy terms.

---

**Fraud Assessment**  
A thorough assessment of the claim has been conducted, and no major fraud indicators have been identified. The fraud status is marked as LOW, indicating a minimal risk of fraudul

In [11]:
result = claim_graph.invoke(initial_state)